In [20]:
import numpy as np
import pandas as pd
import math

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV


### What we need to make this work
Okay so we need to have a df converted to a feather_file with such 
- all the features( parameters ) that we can get
- the y value we want to predict  

In [21]:
df_treated = pd.read_feather('data/treated.feather')

df_treated

Price            Open                                      \
Ticker      crude_oil   US_dollar         gold     silver   
Date                                                        
2025-01-02  71.849998  108.529999  2641.000000  29.260000   
2025-01-03  73.129997  109.199997  2671.100098  30.080000   
2025-01-06  74.050003  108.919998  2652.800049  30.370001   
2025-01-07  73.440002  108.339996  2647.100098  30.447001   
2025-01-08  74.519997  108.620003  2662.899902  30.525000   
...               ...         ...          ...        ...   
2026-10-05  91.769997  101.919998  4169.399902  61.520000   
2026-10-06  89.269997  102.139999  4169.200195  60.919998   
2026-10-07  89.959999  101.889999  4195.000000  60.880001   
2026-10-08  89.000000  102.230003  4139.899902  59.063999   
2026-10-09  91.290001  102.129997  4159.700195  60.669998   

Price                                                                         \
Ticker     3month_treasury_yield    Nasdaq-100 10y_treasury_yield Volatility   
Date                                                                           
2025-01-02                 4.208  21120.050781              4.521  17.209999   
2025-01-03                 4.197  21096.789062              4.545  17.660000   
2025-01-06                 4.188  21551.179688              4.585  16.770000   
2025-01-07                 4.195  21619.890625              4.634  16.480000   
2025-01-08                 4.205  21156.759766              4.705  17.910000   
...                          ...           ...                ...        ...   
2026-10-05                 3.993  30812.759766              5.273  16.240000   
2026-10-06                 4.035  31255.400391              5.273  15.500000   
2026-10-07                 4.037  30976.250000              5.341  15.210000   
2026-10-08                 4.043  30985.230469              5.324  15.680000   
2026-10-09                 4.045  30900.060547              5.236  15.340000   

Price           Close              ...       High        Low              \
Ticker      crude_oil   US_dollar  ... Volatility  crude_oil   US_dollar   
Date                               ...                                     
2025-01-02  73.129997  109.389999  ...  19.500000  71.790001  108.269997   
2025-01-03  73.959999  108.949997  ...  17.940001  72.699997  108.900002   
2025-01-06  73.559998  108.260002  ...  16.870001  73.199997  107.750000   
2025-01-07  74.250000  108.540001  ...  18.900000  73.110001  107.849998   
2025-01-08  73.320000  109.089996  ...  19.500000  73.160004  108.550003   
...               ...         ...  ...        ...        ...         ...   
2026-10-05  89.430000  102.169998  ...  16.379999  88.739998  101.860001   
2026-10-06  89.440002  101.830002  ...  15.540000  86.860001  101.750000   
2026-10-07  88.279999  102.239998  ...  16.010000  87.959999  101.889999   
2026-10-08  91.489998  102.139999  ...  16.459999  88.769997  102.029999   
2026-10-09  91.849998  102.209999  ...  15.340000  90.010002  101.919998   

Price                                                                   \
Ticker             gold     silver 3month_treasury_yield    Nasdaq-100   
Date                                                                     
2025-01-02  2636.100098  29.260000                 4.200  20800.500000   
2025-01-03  2649.699951  29.806000                 4.190  21075.890625   
2025-01-06  2624.600098  29.959999                 4.185  21462.519531   
2025-01-07  2643.500000  30.447001                 4.195  21101.089844   
2025-01-08  2658.399902  30.280001                 4.203  20992.080078   
...                 ...        ...                   ...           ...   
2026-10-05  4150.399902  60.868999                 3.990  30798.419922   
2026-10-06  4130.700195  60.919998                 4.035  31208.199219   
2026-10-07  4091.199951  59.529999                 4.035  30904.460938   
2026-10-08  4128.100098  58.915001                 4.040  30556.400391 

In [22]:
df = df_treated.iloc[:,:32]
y = df_treated["up/down"]

In [23]:
model = RandomForestRegressor(n_estimators= 10, n_jobs=-1, random_state=0)

%time model.fit(df, y)

CPU times: user 48.2 ms, sys: 9.07 ms, total: 57.2 ms
Wall time: 24.9 ms


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",0
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [17]:
model.score(df, y)

0.8232469020097886

In [ ]:
model.

10